In [1]:
from pathlib import Path

import polars as pl
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
ROOT_DIR = Path().cwd().parent
RAW_DATA_DIR = ROOT_DIR / "data" / "raw" / "dataset"

In [3]:
product = pl.read_csv(RAW_DATA_DIR / "product.csv", separator='\t')
label = pl.read_csv(RAW_DATA_DIR / "label.csv", separator='\t')
query = pl.read_csv(RAW_DATA_DIR / "query.csv", separator='\t')

# Метаданные

In [4]:
product.shape, product.schema

((42994, 9),
 Schema([('product_id', Int64),
         ('product_name', String),
         ('product_class', String),
         ('category hierarchy', String),
         ('product_description', String),
         ('product_features', String),
         ('rating_count', Float64),
         ('average_rating', Float64),
         ('review_count', Float64)]))

In [5]:
label.shape, label.schema

((233448, 4),
 Schema([('id', Int64),
         ('query_id', Int64),
         ('product_id', Int64),
         ('label', String)]))

In [6]:
query.shape, query.schema

((480, 3),
 Schema([('query_id', Int64), ('query', String), ('query_class', String)]))

# Пропуски

In [7]:
product.null_count()

product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count
u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,2852,1556,6008,0,9452,9452,9452


In [8]:
label.null_count()

id,query_id,product_id,label
u32,u32,u32,u32
0,0,0,0


In [9]:
query.null_count()

query_id,query,query_class
u32,u32,u32
0,0,6


# Уникальные id

In [10]:
product.select(pl.col("product_id").unique()).shape[0] / product.select(pl.col('product_id')).shape[0]

1.0

In [11]:
query.select(pl.col("query_id").unique()).shape[0] / query.select(pl.col('query_id')).shape[0]

1.0

# Целостность внешних ключей в labels

In [12]:
missing_query_id = (
    label.select("query_id").unique()
    .join(query.select("query_id").unique(), on="query_id", how="anti")
)
missing_query_id

query_id
i64


In [13]:
missing_product_id = (
    label.select("product_id").unique()
    .join(product.select("product_id").unique(), on="product_id", how="anti")
)
missing_product_id

product_id
i64


# Дубликаты в labels

In [14]:
label.filter(
    label.select("query_id", "product_id").is_duplicated()
).unique(subset=["query_id", "product_id"]).sort("query_id")

id,query_id,product_id,label
i64,i64,i64,str
65,0,31306,"""Irrelevant"""
620,4,26878,"""Partial"""
21713,6,22389,"""Partial"""
21712,6,31306,"""Partial"""
1325,12,34349,"""Exact"""
…,…,…,…
58420,486,4699,"""Partial"""
58446,486,9633,"""Partial"""
58438,486,8305,"""Partial"""


# Противоречия в метках

In [15]:
pairs = label.group_by("query_id", "product_id").agg(
    pl.len().alias("n_rows"),
    pl.col("label").n_unique().alias("n_labels"),
    pl.col("label").unique().sort().alias("labels"),
)

conflicts = pairs.filter(pl.col("n_labels") > 1)
conflicts.is_duplicated

<bound method DataFrame.is_duplicated of shape: (14, 5)
┌──────────┬────────────┬────────┬──────────┬───────────────────────────┐
│ query_id ┆ product_id ┆ n_rows ┆ n_labels ┆ labels                    │
│ ---      ┆ ---        ┆ ---    ┆ ---      ┆ ---                       │
│ i64      ┆ i64        ┆ u32    ┆ u32      ┆ list[str]                 │
╞══════════╪════════════╪════════╪══════════╪═══════════════════════════╡
│ 293      ┆ 19210      ┆ 2      ┆ 2        ┆ ["Exact", "Partial"]      │
│ 89       ┆ 24413      ┆ 2      ┆ 2        ┆ ["Exact", "Partial"]      │
│ 258      ┆ 2076       ┆ 2      ┆ 2        ┆ ["Exact", "Partial"]      │
│ 243      ┆ 5766       ┆ 2      ┆ 2        ┆ ["Exact", "Partial"]      │
│ 421      ┆ 29138      ┆ 2      ┆ 2        ┆ ["Exact", "Partial"]      │
│ …        ┆ …          ┆ …      ┆ …        ┆ …                         │
│ 246      ┆ 38402      ┆ 2      ┆ 2        ┆ ["Irrelevant", "Partial"] │
│ 106      ┆ 1990       ┆ 2      ┆ 2        ┆ ["Irreleva

In [16]:
# типы оценок, которые имеют противоречия
conflicts.select("labels").unique(subset=['labels'])
# так то можно просто взять пессимистичную или оптимистичную оценку, 
# но это не совсем корректно, так как мы теряем информацию о том, что есть противоречия

labels
list[str]
"[""Exact"", ""Partial""]"
"[""Irrelevant"", ""Partial""]"


# Распределение меток

In [17]:
label_counts = label.group_by("label").agg(pl.len().alias("n_rows"))

label_counts.with_columns(
    (100 * pl.col("n_rows") / pl.col("n_rows").sum())
    .round(2)
    .alias("pct")
)

label,n_rows,pct
str,u32,f64
"""Exact""",25614,10.97
"""Irrelevant""",61201,26.22
"""Partial""",146633,62.81


In [18]:
# Запрос с максимальным количеством оценок
most_rated = (
    label.group_by("query_id")
    .agg(pl.len().alias("counts"))
    .sort("counts", descending=True)
    .head(1)
)
most_rated.join(query, on="query_id", how="left").select("query_id", "query", "counts")

query_id,query,counts
i64,str,u32
123,"""entrance table""",4329


In [19]:
# Запрос с минимальным количеством оценок
least_rated = (
    label.group_by("query_id")
    .agg(pl.len().alias("counts"))
    .sort("counts", descending=False)
    .head(1)
)
least_rated.join(query, on="query_id", how="left").select("query_id", "query", "counts")

query_id,query,counts
i64,str,u32
366,"""drudge report""",1


In [20]:
# количество запросов без Exact и без Exact/Partial

per_query = label.group_by("query_id").agg(
    (pl.col("label") == "Exact").any().alias("has_exact"),
    pl.col("label").is_in(["Exact", "Partial"]).any().alias("has_exact_or_partial"),
)

count_no_exact = per_query.filter(~pl.col("has_exact")).height
count_no_exact_partial = per_query.filter(~pl.col("has_exact_or_partial")).height

count_no_exact, count_no_exact_partial

(101, 1)

# Оценок на обычный запрос

In [21]:
per_query = label.group_by("query_id").agg(pl.len().alias("n_ratings"))

per_query.select(
    pl.col("n_ratings").min().alias("min"),
    pl.col("n_ratings").median().alias("median"),
    pl.col("n_ratings").mean().alias("mean"),
    pl.col("n_ratings").max().alias("max"),
)

min,median,mean,max
u32,f64,f64,u32
1,216.5,486.35,4329


# Повторяющиеся тексты (нормализация)

In [22]:
# Найти одинаковые тексты запросов после нормализации регистра и пробелов

normalized_query = query.select(
    pl.col("query_id", "query"),
    pl.col("query")
        .str.to_lowercase()
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
        .alias("normalized_query")
)

normalized_query.unique(subset=["normalized_query"]).height, query.height

(480, 480)

# Похожие текста

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

texts = query["query"].to_list()

vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 4),
    lowercase=True,
)

vectors = vectorizer.fit_transform(texts)
similarity = cosine_similarity(vectors)


i, j = np.triu_indices_from(similarity, k=1)
scores = similarity[i, j]
top = np.argsort(-scores)[:100]

query_ids = query["query_id"].to_numpy()

top_100 = pl.DataFrame({
    "query_id_1": query_ids[i[top]],
    "query_id_2": query_ids[j[top]],
    "query_1": [texts[idx] for idx in i[top]],
    "query_2": [texts[idx] for idx in j[top]],
    "similarity": scores[top],
})

with pl.Config(tbl_rows=100):
    display(top_100)

query_id_1,query_id_2,query_1,query_2,similarity
i64,i64,str,str,f64
118,314,"""body pillow case""","""body pillow and case""",0.939891
81,137,"""orren ellis l shape desk""","""orren ellis l shaped desk""",0.917201
26,151,"""leather chairs""","""leather chair""",0.863574
93,388,"""outdoor sectional dining""","""outdoor sectional doning""",0.829653
26,126,"""leather chairs""","""leather dining chairs""",0.814871
121,331,"""upholstered bed""","""upholstered girls bed""",0.814146
391,418,"""writing desk 48""""","""writing desk""",0.812959
32,429,"""dark gray dresser""","""gray dresser""",0.81286
103,231,"""kitchen anti fatigue mats""","""anti fatigue mat""",0.771943


# Длины названий, описаний и характеристик товаров

In [24]:
text_cols = ["product_name", "product_description", "product_features"]

product.select(
    pl.col(text_cols).str.strip_chars().str.len_chars()
).describe()

statistic,product_name,product_description,product_features
str,f64,f64,f64
"""count""",42994.0,36986.0,42994.0
"""null_count""",0.0,6008.0,0.0
"""mean""",40.445085,458.090115,1565.521561
"""std""",20.833983,299.150924,741.984403
"""min""",3.0,6.0,1.0
"""25%""",28.0,259.0,1005.0
"""50%""",37.0,418.0,1494.0
"""75%""",49.0,554.0,2032.0
"""max""",257.0,4060.0,11299.0


# Пустые строки и NaN помимо null

In [25]:
text = pl.col(pl.String).str.strip_chars()

display(product.select((text == "").sum()))
display(product.select((text.str.to_lowercase() == "nan").sum()))
display(product.select(pl.selectors.float().is_nan().sum()))

product_name,product_class,category hierarchy,product_description,product_features
u32,u32,u32,u32,u32
0,0,0,0,0


product_name,product_class,category hierarchy,product_description,product_features
u32,u32,u32,u32,u32
0,0,0,0,0


rating_count,average_rating,review_count
u32,u32,u32
0,0,0
